<a href="https://colab.research.google.com/github/shuangquan-li-con/ECON5200-37499-Applied-Data-Analytics-in-Economics/blob/main/Usrc_decompose_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import numpy as np
import pandas as pd
!pip install ruptures
!pip install fredapi
import ruptures as rpt

from typing import Dict, List, Any, Sequence
from fredapi import Fred
from statsmodels.tsa.seasonal import STL, MSTL
from statsmodels.tsa.stattools import adfuller, kpss


def run_stl(
    series: pd.Series,
    period: int = 12,
    log_transform: bool = True,
    robust: bool = True
):
    """
    Apply STL decomposition with optional log transform.

    Parameters
    ----------
    series : pd.Series
        Time series with DatetimeIndex.
    period : int, default=12
        Seasonal period.
    log_transform : bool, default=True
        Whether to log-transform the series before decomposition.
    robust : bool, default=True
        Whether to use robust STL fitting.

    Returns
    -------
    STL result
        Fitted STL result object.
    """
    if not isinstance(series, pd.Series):
        raise TypeError("series must be a pandas Series")

    series = series.dropna()

    if len(series) == 0:
        raise ValueError("series is empty after dropping missing values")

    if not isinstance(series.index, pd.DatetimeIndex):
        raise ValueError("series must have a DatetimeIndex")

    if log_transform:
        if (series <= 0).any():
            raise ValueError("series must be strictly positive when log_transform=True")
        series = np.log(series)

    return STL(series, period=period, robust=robust).fit()


def run_mstl(
    series: pd.Series,
    periods: Sequence[int]
):
    """
    Apply MSTL decomposition for multiple seasonal periods.

    Parameters
    ----------
    series : pd.Series
        Time series with DatetimeIndex.
    periods : sequence of int
        Seasonal periods, e.g. [24, 168] for hourly daily + weekly seasonality.

    Returns
    -------
    MSTL result
        Fitted MSTL result object.
    """
    if not isinstance(series, pd.Series):
        raise TypeError("series must be a pandas Series")

    series = series.dropna()

    if len(series) == 0:
        raise ValueError("series is empty after dropping missing values")

    if not isinstance(series.index, pd.DatetimeIndex):
        raise ValueError("series must have a DatetimeIndex")

    if not periods:
        raise ValueError("periods must contain at least one seasonal period")

    return MSTL(series, periods=list(periods)).fit()


def test_stationarity(
    series: pd.Series,
    alpha: float = 0.05,
    regression: str = "ct"
) -> Dict[str, Any]:
    """
    Run ADF and KPSS tests and return a 2x2 verdict.

    Parameters
    ----------
    series : pd.Series
        Input time series.
    alpha : float, default=0.05
        Significance level.
    regression : str, default='ct'
        Deterministic terms for both tests.
        Common options:
        - 'c'  : constant
        - 'ct' : constant + trend

    Returns
    -------
    dict
        Dictionary with ADF/KPSS statistics, p-values, and verdict.
    """
    if not isinstance(series, pd.Series):
        raise TypeError("series must be a pandas Series")

    series = series.dropna()

    if len(series) == 0:
        raise ValueError("series is empty after dropping missing values")

    if not isinstance(series.index, pd.DatetimeIndex):
        raise ValueError("series must have a DatetimeIndex")

    adf_stat, adf_p, _, _, _, _ = adfuller(series, regression=regression)
    kpss_stat, kpss_p, _, _ = kpss(series, regression=regression, nlags="auto")

    adf_reject = adf_p < alpha
    kpss_reject = kpss_p < alpha

    if adf_reject and not kpss_reject:
        verdict = "stationary"
    elif (not adf_reject) and kpss_reject:
        verdict = "non-stationary"
    elif adf_reject and kpss_reject:
        verdict = "contradictory"
    else:
        verdict = "inconclusive"

    return {
        "adf_stat": adf_stat,
        "adf_p": adf_p,
        "kpss_stat": kpss_stat,
        "kpss_p": kpss_p,
        "verdict": verdict
    }


def detect_breaks(
    series: pd.Series,
    pen: float = 10,
    model: str = "rbf"
) -> List[pd.Timestamp]:
    """
    Detect structural breaks using PELT.

    Parameters
    ----------
    series : pd.Series
        Input time series with DatetimeIndex.
    pen : float, default=10
        Penalty value. Higher means fewer breakpoints.
    model : str, default='rbf'
        Cost model for ruptures.

    Returns
    -------
    list[pd.Timestamp]
        Break dates.
    """
    if not isinstance(series, pd.Series):
        raise TypeError("series must be a pandas Series")

    series = series.dropna()

    if len(series) == 0:
        raise ValueError("series is empty after dropping missing values")

    if not isinstance(series.index, pd.DatetimeIndex):
        raise ValueError("series must have a DatetimeIndex")

    algo = rpt.Pelt(model=model).fit(series.values)
    breakpoints = algo.predict(pen=pen)

    break_dates = []
    for bp in breakpoints:
        if bp < len(series):
            break_dates.append(series.index[bp])

    return break_dates


def block_bootstrap_trend(
    series: pd.Series,
    period: int,
    n_bootstrap: int = 200,
    block_size: int = 8,
    log_transform: bool = True,
    robust: bool = True,
    ci: float = 0.90
) -> Dict[str, Any]:
    """
    Estimate STL trend uncertainty using moving block bootstrap.

    Why block bootstrap instead of i.i.d. bootstrap?
    ------------------------------------------------
    Time-series residuals are usually autocorrelated. Block bootstrap
    preserves short-run dependence within each sampled block, while
    i.i.d. bootstrap destroys that structure.

    Parameters
    ----------
    series : pd.Series
        Input time series.
    period : int
        STL seasonal period.
    n_bootstrap : int, default=200
        Number of bootstrap replications.
    block_size : int, default=8
        Length of each sampled residual block.
    log_transform : bool, default=True
        Whether to log-transform before STL.
    robust : bool, default=True
        Whether to use robust STL.
    ci : float, default=0.90
        Confidence level.

    Returns
    -------
    dict
        Original trend, lower band, upper band, fitted decomposition, and transformed series.
    """
    if not isinstance(series, pd.Series):
        raise TypeError("series must be a pandas Series")

    series = series.dropna()

    if len(series) == 0:
        raise ValueError("series is empty after dropping missing values")

    if not isinstance(series.index, pd.DatetimeIndex):
        raise ValueError("series must have a DatetimeIndex")

    if log_transform:
        if (series <= 0).any():
            raise ValueError("series must be strictly positive when log_transform=True")
        transformed = np.log(series)
    else:
        transformed = series.copy()

    base = STL(transformed, period=period, robust=robust).fit()

    trend = base.trend
    seasonal = base.seasonal
    resid = base.resid.values

    n = len(transformed)
    boot_trends = np.zeros((n_bootstrap, n))

    for b in range(n_bootstrap):
        boot_resid = np.zeros(n)
        idx = 0

        while idx < n:
            start = np.random.randint(0, n - block_size + 1)
            block = resid[start:start + block_size]
            end = min(idx + block_size, n)
            boot_resid[idx:end] = block[:end - idx]
            idx = end

        boot_series = pd.Series(
            trend.values + seasonal.values + boot_resid,
            index=transformed.index
        )

        boot_fit = STL(boot_series, period=period, robust=robust).fit()
        boot_trends[b, :] = boot_fit.trend.values

    alpha = 1 - ci
    lower_q = 100 * (alpha / 2)
    upper_q = 100 * (1 - alpha / 2)

    lower = np.percentile(boot_trends, lower_q, axis=0)
    upper = np.percentile(boot_trends, upper_q, axis=0)

    return {
        "transformed_series": transformed,
        "fit": base,
        "trend": trend,
        "lower": pd.Series(lower, index=transformed.index),
        "upper": pd.Series(upper, index=transformed.index),
        "boot_trends": boot_trends
    }


def fetch_fred_series(
    series_id: str,
    api_key: str,
    observation_start: str | None = None
) -> pd.Series:
    """
    Fetch a FRED time series.

    Parameters
    ----------
    series_id : str
        FRED series ID.
    api_key : str
        User's FRED API key.
    observation_start : str | None
        Optional start date like '2000-01-01'.

    Returns
    -------
    pd.Series
        FRED series with DatetimeIndex.
    """
    fred = Fred(api_key=api_key)
    series = fred.get_series(series_id, observation_start=observation_start)
    series = series.dropna()
    series.index = pd.DatetimeIndex(series.index)
    return series


if __name__ == "__main__":
    np.random.seed(42)
    print("decompose.py loaded successfully.")

    # Self-test for run_stl
    idx = pd.date_range("2015-01-01", periods=120, freq="MS")
    t = np.arange(120)
    y = pd.Series(
        100 * np.exp(0.01 * t) * (1 + 0.2 * np.sin(2 * np.pi * t / 12)) * np.exp(np.random.normal(0, 0.03, 120)),
        index=idx
    )
    stl_fit = run_stl(y, period=12, log_transform=True)
    print("run_stl self-test passed:", hasattr(stl_fit, "trend"))

    # Self-test for test_stationarity
    stationary_series = pd.Series(np.random.normal(size=200), index=pd.date_range("2000-01-01", periods=200, freq="MS"))
    nonstationary_series = stationary_series.cumsum()
    print("stationary verdict:", test_stationarity(stationary_series, regression="c")["verdict"])
    print("nonstationary verdict:", test_stationarity(nonstationary_series, regression="c")["verdict"])

    # Self-test for detect_breaks
    break_idx = pd.date_range("2010-01-01", periods=240, freq="MS")
    break_series = pd.Series(
        np.r_[np.random.normal(0, 1, 80), np.random.normal(3, 1, 80), np.random.normal(-2, 1, 80)],
        index=break_idx
    )
    print("detected breaks:", detect_breaks(break_series, pen=8))

    # Self-test for run_mstl
    n_hours = 24 * 7 * 8
    h = np.arange(n_hours)
    hourly_idx = pd.date_range("2024-01-01", periods=n_hours, freq="h")
    hourly_series = pd.Series(
        500 + 0.01 * h + 80 * np.sin(2 * np.pi * h / 24) + 40 * np.sin(2 * np.pi * h / 168) + np.random.normal(0, 15, n_hours),
        index=hourly_idx
    )
    mstl_fit = run_mstl(hourly_series, periods=[24, 168])
    print("run_mstl self-test passed:", hasattr(mstl_fit, "seasonal"))

    # Self-test for block bootstrap
    boot = block_bootstrap_trend(y, period=12, n_bootstrap=20, block_size=6, log_transform=True)
    print("block_bootstrap_trend self-test passed:", "lower" in boot and "upper" in boot)

decompose.py loaded successfully.
run_stl self-test passed: True
stationary verdict: stationary
nonstationary verdict: non-stationary
detected breaks: [Timestamp('2016-09-01 00:00:00'), Timestamp('2023-05-01 00:00:00')]


/tmp/ipykernel_7751/451428085.py:130: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_stat, kpss_p, _, _ = kpss(series, regression=regression, nlags="auto")
/tmp/ipykernel_7751/451428085.py:130: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_stat, kpss_p, _, _ = kpss(series, regression=regression, nlags="auto")


run_mstl self-test passed: True
block_bootstrap_trend self-test passed: True
